In [3]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

workoutId = '13-09-2025_190357'
con = duckdb.connect("./hr_data/database.duckdb")
df = con.execute(f"SELECT * FROM timeseries WHERE starts_with(workoutId, '{workoutId}')").fetchdf()
con.close()

# Get the first and last X values (Time)
first_time = df['Time'].iloc[0]
last_time = df['Time'].iloc[-1]
max_idx = df['HR (bpm)'].idxmax()
max_time = df.loc[max_idx, 'Time']
max_hr = df.loc[max_idx, 'HR (bpm)']

# Plot using Plotly
fig = px.line(df, y='HR (bpm)', x='Time', title='Heart Rate')

# Get the minimum HR value
min_hr = df['HR (bpm)'].min()

# Load and sort CSV by HR ascending
df = pd.read_csv('./hr_data/zones.csv')
df = df.sort_values('HR').reset_index(drop=True)

# Build background shapes and annotations
shapes = []
annotations = []
previous_hr = min_hr - 5

colors = ['lightgray', 'green', 'lightblue', 'yellow', 'lightcoral', 'red', 'purple']
# Truncate the colors list to the number of rows in the zones df
colors = colors[:len(df)]

for i, row in df.iterrows():
    y0 = previous_hr
    y1 = row['HR']
    color = colors[i]

    # Background rectangle
    shapes.append(dict(
        type="rect",
        xref="paper", yref="y",
        x0=0, x1=1,
        y0=y0, y1=y1,
        fillcolor=color,
        opacity=0.5,
        layer="below",
        line_width=0,
    ))

    previous_hr = y1

# Update layout with shapes and annotations
fig.update_layout(shapes=shapes, annotations=annotations)

# Hover mode - the vertical line that follows the cursor
fig.update_layout(
    hovermode='x',  # or 'x unified' for grouped tooltip
    xaxis=dict(
        showspikes=True,
        spikemode='across',
        spikesnap='cursor',
        showline=True,
        spikethickness=1,
        spikecolor="gray",
        spikedash="solid",
        tickvals=[first_time, last_time],
        ticktext=[str(first_time), str(last_time)],
    ),
    yaxis=dict(
        range=[80, None],
        dtick=10  # Show ticks every 10 bpm
    )
)

# Optional: consistent hover label
fig.update_traces(
    hovertemplate='Time: %{x}<br>HR: %{y} bpm',
    mode='lines'  # adds points for better hover visibility
)

#Step 3: Add marker for max point
fig.add_trace(go.Scatter(
    x=[max_time],
    y=[max_hr],
    mode='markers+text',
    marker=dict(color='red', size=10, symbol='circle'),
    text=[f'Max: {max_hr} bpm'],
    textposition='top center',
    showlegend=False
))

# Convert to a dict for Zones 1 through N
zone_colors = {i + 1: color for i, color in enumerate(colors)}

for zone, color in zone_colors.items():
    fig.add_trace(go.Scatter(
        x=[None], y=[None],  # No actual data
        mode='markers',
        marker=dict(size=10),
        name=f'Zone {zone}',
        legendgroup=f'Zone {zone}',
        showlegend=True,
        marker_color=color
    ))

fig.show()

In [4]:
from plotly.subplots import make_subplots

con = duckdb.connect("./hr_data/database.duckdb")

def query_builder(workoutId, table_name):
    return f"SELECT * FROM {table_name}" if workoutId == '' else f"SELECT * FROM {table_name} WHERE starts_with(workoutId, '{workoutId}')"

workoutId = '13-09-2025_190357'
hr_df = con.execute(query_builder(workoutId, 'timeseries')).fetchdf()
meta_df = con.execute(query_builder(workoutId, 'workout_metadata')).fetchdf()
calories_df = con.execute("SELECT * FROM calories_per_hr").fetchdf()
con.close()

# 1.Get the minimum HR value
min_hr = hr_df['HR (bpm)'].min()

# 3. Build zone intervals: each zone is between previous HR and current HR
zone_bounds = []
previous_hr = min_hr-5

zones_df = pd.read_csv('./hr_data/zones.csv')
zones_df = zones_df.sort_values('HR').reset_index(drop=True)

# Generating the zone boundaries. E.g. Zone 2 = highest in Zone 1 + 1 to highest in Zone 2
for i, row in zones_df.iterrows():
    zone_bounds.append((previous_hr, row['HR'], row['Zone']))
    previous_hr = row['HR']

# 3. Classify each HR value into a zone
def classify_zone(hr_value):
    for lower, upper, zone in zone_bounds:
        if lower <= hr_value < upper:
            return zone
    return zone_bounds[-1][2]  # Assign to last zone if HR >= max

# Calculate the zones for each HR value and the calories per second for that HR
# HR is BPM measured each second.
hr_df['Zone'] = hr_df['HR (bpm)'].apply(classify_zone)
hr_df['Calories_Second'] = hr_df['HR (bpm)'].apply(
    lambda hr: calories_df.loc[calories_df['HR'] == hr, 'Calories_Second'].values[0] if hr in calories_df['HR'].values else 0)

# 4. Calculate percentage of time in each zone
zone_counts = hr_df['Zone'].value_counts().sort_index()
zone_percentages = (zone_counts / len(hr_df) * 100).round(2)
# Just to validate the DF here
# display(zone_counts)

# 5. Display result
# Calculate total calories burned
total_calories = hr_df['Calories_Second'].sum()
print(f"Total calories burned: {total_calories:.2f} kcal")
# Reading the sum of all metadata calories as the SELECT statement might return multiple rows
print(f"Total calories burned from metadata source: {meta_df['Calories'].sum()} kcal")
print("Percentage of time in each HR zone:")
pie_df = zone_percentages.to_frame(name="Percentage (%)")

# The reset_index moves the index to a column
colors = ['lightgray', 'darkgreen', 'lightblue', 'yellow', 'lightcoral']
pie_df = pie_df.reset_index()
pie_df['Zone'] = pie_df.apply(
    lambda row: f"{int(row['Zone'])}: {row['Percentage (%)']:.1f}%", axis=1
)
pie_df['Time'] = zone_counts.reset_index(drop=True).apply(
    lambda seconds: f"{seconds // 3600:02}:{(seconds % 3600) // 60:02}:{seconds % 60:02}"
)

# Create a subplot with two columns: one for the pie chart and one for the table
fig = make_subplots(
    rows=1, cols=2, 
    column_widths=[0.6, 0.4],  # Adjust column widths
    specs=[[{"type": "domain"}, {"type": "table"}]],  # Specify chart types
    subplot_titles=["Time Spent in Each HR Zone", "Zone Details"]
)

fig.add_trace(
    go.Pie(
        labels=pie_df['Zone'],
        values=pie_df['Percentage (%)'],
        hole=0.4,  # Optional: donut chart
        marker=dict(colors=colors[:len(pie_df)]),
    ),
    row=1, col=1
)

# Add the table to the second column
fig.add_trace(
    go.Table(
        header=dict(values=["Zone", "Time"], align='center', font=dict(size=12, color='white'), fill_color='darkblue'),
        cells=dict(values=[pie_df['Zone'], pie_df['Time']], align='center', font=dict(size=10), fill_color='lightgray')
    ),
    row=1, col=2
)

# Adjust layout to fit both the pie chart and table
fig.update_layout(
    title_text="Time Spent in Each HR Zone",
    title_x=0.5,  # Center the title
    margin=dict(l=50, r=50, t=50, b=50)  # Adjust margins for better fit
)

fig.show()

Total calories burned: 60.78 kcal
Total calories burned from metadata source: 67 kcal
Percentage of time in each HR zone:
